<a href="https://colab.research.google.com/github/stepanmouratoglou-a11y/Premier-League-Predictions/blob/main/PremierLeague.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Importing the Libraries**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# **Importing the dataset & Feature Engineering**
##### Here , I am creating some new categories such as

*I drop some columns that will not help on predicting future outcomes , such as Division,Home and Away Red Cards and finally the **time** that the match took place*


*   Average Goals **scored** from the last 5 games
*   Average Goals **conceded** from the last 5 games
*   Average shots from the last 5 games
* Average shots **conceded** from the last 5 games
* Respectively for the shots **on target** from the last 5 games
* Average corners made last 5 games
* Average corners **conceded** last 5 games
* Wins last 5 games
* Losses Last 5 games
* Draws Last 5 games
* Goal Difference

Also, I create a DataFrame which I turn into csv , to visualize all the data and the analytics at the EDA.ipynb file

In [2]:
dataset=pd.read_csv('E0 (1).csv').iloc[:,:24]

dataset=dataset.drop(columns=['Div','HR','AR','Time','Referee'])

dataset['Date'] = pd.to_datetime(dataset['Date'], format='%d/%m/%Y')
dataset['Month']=dataset['Date'].dt.month
dataset['Day']=dataset['Date'].dt.dayofweek

home_matches=dataset[['Date','HomeTeam']].rename(columns={'HomeTeam':'Team'})
away_matches=dataset[['Date','AwayTeam']].rename(columns={'AwayTeam':'Team'})

matches=pd.concat((home_matches,away_matches)).sort_values(['Team','Date'])

matches['Days_Rest']=matches.groupby('Team')['Date'].diff().dt.days
matches['Days_Rest']=matches['Days_Rest'].fillna(15)

dataset=dataset.merge(matches, left_on=['Date', 'HomeTeam'], right_on=['Date', 'Team'], how='left')
dataset=dataset.rename(columns={'Days_Rest': 'Home_Days_Rest'}).drop(columns=['Team'])

dataset=dataset.merge(matches, left_on=['Date', 'AwayTeam'], right_on=['Date', 'Team'], how='left')
dataset=dataset.rename(columns={'Days_Rest': 'Away_Days_Rest'}).drop(columns=['Team'])

dataset=dataset.sort_values('Date').reset_index(drop=True)


home_stats = dataset[['Date', 'HomeTeam', 'FTHG', 'FTAG','HS','AS','HST',
                      'AST','HC','AC','HF','AF','HY','AY']].copy()
home_stats = home_stats.rename(columns={
    'HomeTeam':'Team',
    'FTHG':'Scored',
    'FTAG':'Conceded',
    'HS':'Shots_made',
    'AS':'Shots_Conceded',
    'HST':'Shots_ontarget_made',
    'AST':'Shots_ontarget_conceded',
    'HC':'Corners_made',
    'AC':'Corners_conceded',
    'HF':'Fouls_commited',
    'AF':'Fouls_suffered',
    'HY':'Yellow_Cards',
    'AY':'Opposing_Yellow_Cards'
})
home_stats['Wins']= (dataset['FTR']=='H').astype(int)
home_stats['Draws']=(dataset['FTR']=='D').astype(int)
home_stats['Losses']=(dataset['FTR']=='A').astype(int)

away_stats = dataset[['Date','AwayTeam','FTAG','FTHG','HS','AS','HST',
                      'AST','HC','AC','HF','AF','HY','AY']].copy()
away_stats = away_stats.rename(columns={
    'AwayTeam':'Team',
    'FTAG':'Scored',
    'FTHG':'Conceded',
    'HS':'Shots_Conceded',#This does not refer to the goals conceded , it contains the amount of all shots against the team
    'AS':'Shots_made',
    'HST':'Shots_ontarget_conceded',
    'AST':'Shots_ontarget_made',
    'HC':'Corners_conceded',#This does not refer to the goals conceded from corners , but the amount of corners the opposing team got
    'AC':'Corners_made',
    'HF':'Fouls_suffered',
    'AF':'Fouls_commited',
    'HY':'Opposing_Yellow_Cards',
    'AY':'Yellow_Cards'
})

away_stats['Wins']=(dataset['FTR']=='A').astype(int)
away_stats['Draws']=(dataset['FTR']=='D').astype(int)
away_stats['Losses']=(dataset['FTR']=='H').astype(int)

all_stats=pd.concat([home_stats,away_stats])
team_performance=all_stats.drop(columns=['Date']).groupby('Team').sum().reset_index()
total_games=team_performance['Wins']+team_performance['Draws']+team_performance['Losses']


team_performance['Goal_Difference']=team_performance['Scored']-team_performance['Conceded']
team_performance['Shots_Per_Game']=team_performance['Shots_made']/total_games
team_performance['Corners_Per_Game']=team_performance['Corners_made']/total_games
team_performance['SoT_Per_Game']=team_performance['Shots_ontarget_made']/total_games
team_performance['Goals_Per_Game']=team_performance['Scored']/total_games
team_performance['Goals_Conceded_Per_Game']=team_performance['Conceded']/total_games
team_performance['Fouls_commited_Per_Game']=team_performance['Fouls_commited']/total_games
team_performance['Shots_Conceded_Per_Game']=team_performance['Shots_Conceded']/total_games
team_performance['Yellow_Cards_Per_Game']=team_performance['Yellow_Cards']/total_games

team_stats = pd.concat([home_stats, away_stats]).sort_values(['Team', 'Date']).reset_index(drop=True)


team_stats['Avg_Scored_Last_5'] = team_stats.groupby('Team')['Scored'].\
transform(lambda x: x.shift(1).rolling(window=5).mean())
team_stats['Avg_Conceded_Last_5'] = team_stats.groupby('Team')['Conceded'].\
transform(lambda x: x.shift(1).rolling(window=5).mean())

team_stats['Wins_Last_5']=team_stats.groupby('Team')['Wins'].transform\
 (lambda x: x.shift(1).rolling(window=5).sum())
team_stats['Losses_Last_5']=team_stats.groupby('Team')['Losses'].transform\
 (lambda x:x.shift(1).rolling(window=5).sum())
team_stats['Draws_Last_5']=team_stats.groupby('Team')['Draws'].transform\
 (lambda x: x.shift(1).rolling(window=5).sum())

team_stats['Avg_Shots_Last_5']=team_stats.groupby('Team')['Shots_made'].transform\
 (lambda x: x.shift(1).rolling(window=5).mean())
team_stats['Avg_Shots_Conceded_Last_5']=team_stats.groupby('Team')['Shots_Conceded'].transform\
 (lambda x: x.shift(1).rolling(window=5).mean())

team_stats['Avg_shots_ontarget_Last_5']=team_stats.groupby('Team')['Shots_ontarget_made'].\
transform(lambda x: x.shift(1).rolling(window=5).mean())
team_stats['Avg_shots_ontarget_Conceded_Last_5']=team_stats.groupby('Team')['Shots_ontarget_conceded'].\
transform(lambda x: x.shift(1).rolling(window=5).mean())

team_stats['Avg_corners_Last_5']=team_stats.groupby('Team')['Corners_made'].transform\
 (lambda x: x.shift(1).rolling(window=5).mean())
team_stats['Avg_corners_conceded_Last_5']=team_stats.groupby('Team')['Corners_conceded'].transform\
 (lambda x: x.shift(1).rolling(window=5).mean())

team_stats=team_stats.fillna(0)

dataset=dataset.merge(
    team_stats[['Date', 'Team', 'Avg_Scored_Last_5', 'Avg_Conceded_Last_5','Avg_Shots_Last_5',
                'Avg_Shots_Conceded_Last_5','Avg_shots_ontarget_Last_5','Avg_shots_ontarget_Conceded_Last_5',
                'Avg_corners_Last_5','Avg_corners_conceded_Last_5','Wins_Last_5','Draws_Last_5',
                'Losses_Last_5']],
    left_on=['Date', 'HomeTeam'],
    right_on=['Date', 'Team'],
    how='left'
).rename(columns={
    'Avg_Scored_Last_5': 'Home_Avg_Scored_5',
    'Avg_Conceded_Last_5': 'Home_Avg_Conceded_5',
    'Avg_Shots_Last_5':'Home_Avg_Shots_Last_5',
    'Avg_Shots_Conceded_Last_5':'Home_Ang_Shots_Conceded_Last_5',
    'Avg_shots_ontarget_Last_5':'Home_Avg_shots_ontarget_Last_5',
    'Avg_shots_ontarget_Conceded_Last_5':'Home_Avg_shots_ontarget_Conceded_Last_5',
    'Avg_corners_Last_5': 'Home_Avg_corners_Last_5',
    'Avg_corners_conceded_Last_5':'Home_Avg_corners_conceded_Last_5',
    'Wins_Last_5':'Home_Wins_Last_5',
    'Losses_Last_5':'Home_Losses_Last_5',
    'Draws_Last_5':'Home_Draws_Last_5'
}).drop(columns=['Team'])

dataset=dataset.merge(
    team_stats[['Date', 'Team', 'Avg_Scored_Last_5', 'Avg_Conceded_Last_5','Avg_Shots_Last_5',
                'Avg_Shots_Conceded_Last_5','Avg_shots_ontarget_Last_5','Avg_shots_ontarget_Conceded_Last_5',
                'Avg_corners_Last_5','Avg_corners_conceded_Last_5','Wins_Last_5','Draws_Last_5',
                'Losses_Last_5']],
    left_on=['Date', 'AwayTeam'],
    right_on=['Date', 'Team'],
    how='left'
).rename(columns={
    'Avg_Scored_Last_5': 'Away_Avg_Scored_5',
    'Avg_Conceded_Last_5': 'Away_Avg_Conceded_5',
    'Avg_Shots_Last_5':'Away_Avg_Shots_Last_5',
    'Avg_Shots_Conceded_Last_5':'Away_Ang_Shots_Conceded_Last_5',
    'Avg_shots_ontarget_Last_5':'Away_Avg_shots_ontarget_Last_5',
    'Avg_shots_ontarget_Conceded_Last_5':'Away_Avg_shots_ontarget_Conceded_Last_5',
    'Avg_corners_Last_5': 'Away_Avg_corners_Last_5',
    'Avg_corners_conceded_Last_5':'Away_Avg_corners_conceded_Last_5',
    'Wins_Last_5':'Away_Wins_Last_5',
    'Losses_Last_5':'Away_Losses_Last_5',
    'Draws_Last_5':'Away_Draws_Last_5'
}).drop(columns=['Team'])


In [3]:
team_performance.to_csv('Team_Performances.csv',index=False)

## **Importing the dataset again after feature engineering.**
*More specifically , I remove the **Date** column , as the column was formatted better at the step above*.

*Also , I remove the categories "FullTime Away Goals" ,"FullTime Home Goals", "Home Scored", "Away Scored" to avoid **Data Leakage***

Note:These categories are not important now, as new categories were created above eg."Average goals last 5 games"

In [4]:
team_performance.head()

,Team,Scored,Conceded,Shots_made,Shots_Conceded,Shots_ontarget_made,Shots_ontarget_conceded,Corners_made,Corners_conceded,Fouls_commited,...,Losses,Goal_Difference,Shots_Per_Game,Corners_Per_Game,SoT_Per_Game,Goals_Per_Game,Goals_Conceded_Per_Game,Fouls_commited_Per_Game,Shots_Conceded_Per_Game,Yellow_Cards_Per_Game
0,Arsenal,61,22,454,242,153,72,181,103,316,...,3,39,14.645161,5.838710,4.935484,1.967742,0.709677,10.193548,7.806452,1.290323
1,Aston Villa,42,37,397,395,135,132,166,148,310,...,9,5,12.806452,5.354839,4.354839,1.354839,1.193548,10.000000,12.741935,1.516129
2,Bournemouth,46,48,435,393,149,140,177,165,367,...,7,-2,14.032258,5.709677,4.806452,1.483871,1.548387,11.838710,12.677419,2.290323
3,Brentford,46,42,320,390,122,122,148,157,329,...,11,4,10.322581,4.774194,3.935484,1.483871,1.354839,10.612903,12.580645,1.870968
4,Brighton,41,37,398,378,138,122,147,150,374,...,10,4,12.838710,4.741935,4.451613,1.322581,1.193548,12.064516,12.193548,2.548387


In [5]:
dataset=dataset.drop(columns=['Date','FTAG','FTHG','HTR','HS','AS','Day','Month','HY','AY',
                              'HTHG','HTAG','HST','AST','HF','AF','HC','AC'])
X=dataset.drop(columns=['FTR']).values
y=dataset['FTR'].values

In [6]:
dataset.head()

,HomeTeam,AwayTeam,FTR,Home_Days_Rest,Away_Days_Rest,Home_Avg_Scored_5,Home_Avg_Conceded_5,Home_Avg_Shots_Last_5,Home_Ang_Shots_Conceded_Last_5,Home_Avg_shots_ontarget_Last_5,...,Away_Avg_Conceded_5,Away_Avg_Shots_Last_5,Away_Ang_Shots_Conceded_Last_5,Away_Avg_shots_ontarget_Last_5,Away_Avg_shots_ontarget_Conceded_Last_5,Away_Avg_corners_Last_5,Away_Avg_corners_conceded_Last_5,Away_Wins_Last_5,Away_Draws_Last_5,Away_Losses_Last_5
0,Liverpool,Bournemouth,H,15.0,15.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Aston Villa,Newcastle,D,15.0,15.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Brighton,Fulham,D,15.0,15.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Sunderland,West Ham,H,15.0,15.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Tottenham,Burnley,H,15.0,15.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
dataset.shape

(309, 27)

## **Checking for missing values**

In [8]:
indep_cols=dataset.iloc[:,:-1]
missing_values=list(dataset.columns.get_loc(cols) for cols in indep_cols if dataset[cols].isnull().sum()>0 )

In [9]:
print(missing_values)

[]


In [10]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 27 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   HomeTeam                                 309 non-null    object 
 1   AwayTeam                                 309 non-null    object 
 2   FTR                                      309 non-null    object 
 3   Home_Days_Rest                           309 non-null    float64
 4   Away_Days_Rest                           309 non-null    float64
 5   Home_Avg_Scored_5                        309 non-null    float64
 6   Home_Avg_Conceded_5                      309 non-null    float64
 7   Home_Avg_Shots_Last_5                    309 non-null    float64
 8   Home_Ang_Shots_Conceded_Last_5           309 non-null    float64
 9   Home_Avg_shots_ontarget_Last_5           309 non-null    float64
 10  Home_Avg_shots_ontarget_Conceded_Last_5  309 non-n

## **Saving the categorical feature's indexes to apply Target Encoding Later**

In [11]:
categorical_features=[]
for cols in dataset.columns:
  if cols=='FTR':
    continue
  if dataset[cols].dtype =='object':
    categorical_features.append(dataset.columns.get_loc(cols))
print(categorical_features)

[0, 1]


In [12]:
print(y)

['H' 'D' 'D' 'H' 'H' 'A' 'D' 'H' 'A' 'H' 'A' 'A' 'H' 'H' 'H' 'H' 'D' 'H'
 'D' 'A' 'H' 'H' 'H' 'A' 'A' 'D' 'H' 'A' 'H' 'A' 'D' 'H' 'H' 'D' 'D' 'H'
 'H' 'A' 'H' 'A' 'H' 'D' 'D' 'A' 'A' 'H' 'H' 'D' 'D' 'D' 'H' 'A' 'H' 'D'
 'H' 'A' 'D' 'A' 'H' 'D' 'H' 'A' 'H' 'H' 'H' 'H' 'H' 'H' 'D' 'A' 'H' 'H'
 'D' 'A' 'H' 'A' 'H' 'A' 'A' 'A' 'H' 'A' 'H' 'H' 'H' 'A' 'A' 'H' 'H' 'H'
 'H' 'A' 'H' 'H' 'D' 'A' 'H' 'H' 'H' 'D' 'D' 'H' 'H' 'D' 'H' 'D' 'H' 'H'
 'H' 'H' 'A' 'A' 'H' 'H' 'D' 'A' 'H' 'H' 'A' 'A' 'H' 'H' 'H' 'A' 'A' 'A'
 'H' 'A' 'A' 'D' 'A' 'A' 'D' 'D' 'A' 'H' 'A' 'H' 'A' 'D' 'D' 'H' 'H' 'H'
 'D' 'H' 'H' 'D' 'A' 'A' 'H' 'H' 'A' 'H' 'D' 'A' 'A' 'H' 'H' 'D' 'D' 'D'
 'D' 'H' 'A' 'A' 'A' 'H' 'H' 'H' 'H' 'A' 'H' 'D' 'A' 'H' 'A' 'H' 'D' 'A'
 'H' 'D' 'D' 'D' 'A' 'A' 'D' 'D' 'D' 'D' 'H' 'H' 'H' 'A' 'D' 'D' 'D' 'A'
 'D' 'H' 'A' 'H' 'H' 'D' 'D' 'H' 'D' 'D' 'H' 'D' 'D' 'A' 'D' 'H' 'H' 'H'
 'H' 'D' 'A' 'D' 'H' 'H' 'H' 'H' 'D' 'A' 'A' 'A' 'A' 'D' 'H' 'H' 'A' 'D'
 'A' 'A' 'H' 'D' 'D' 'H' 'H' 'A' 'A' 'A' 'H' 'D' 'H

## **Applying Label Encoding on the depentent variable - y**
* Before this , the y-dependent variable has values:
"H" for Home win,
"A" for Away win,
"D" for a Draw

In [13]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)

In [14]:
print(y)

[2 1 1 2 2 0 1 2 0 2 0 0 2 2 2 2 1 2 1 0 2 2 2 0 0 1 2 0 2 0 1 2 2 1 1 2 2
 0 2 0 2 1 1 0 0 2 2 1 1 1 2 0 2 1 2 0 1 0 2 1 2 0 2 2 2 2 2 2 1 0 2 2 1 0
 2 0 2 0 0 0 2 0 2 2 2 0 0 2 2 2 2 0 2 2 1 0 2 2 2 1 1 2 2 1 2 1 2 2 2 2 0
 0 2 2 1 0 2 2 0 0 2 2 2 0 0 0 2 0 0 1 0 0 1 1 0 2 0 2 0 1 1 2 2 2 1 2 2 1
 0 0 2 2 0 2 1 0 0 2 2 1 1 1 1 2 0 0 0 2 2 2 2 0 2 1 0 2 0 2 1 0 2 1 1 1 0
 0 1 1 1 1 2 2 2 0 1 1 1 0 1 2 0 2 2 1 1 2 1 1 2 1 1 0 1 2 2 2 2 1 0 1 2 2
 2 2 1 0 0 0 0 1 2 2 0 1 0 0 2 1 1 2 2 0 0 0 2 1 2 0 0 0 1 0 0 1 2 0 2 1 0
 1 1 2 1 1 0 1 0 2 0 0 0 2 1 0 2 0 0 2 2 2 2 1 2 0 2 1 0 2 0 0 0 1 0 2 0 1
 1 1 1 2 1 1 2 2 2 1 2 0 0]


## **Initializing the training and test set**

In [15]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [16]:
print(X_test)

[['Aston Villa' 'Chelsea' 5.0 ... 2.0 2.0 1.0]
 ['Leeds' 'Everton' 15.0 ... 0.0 0.0 0.0]
 ['Newcastle' 'Arsenal' 7.0 ... 3.0 1.0 1.0]
 ...
 ["Nott'm Forest" 'Arsenal' 11.0 ... 4.0 1.0 0.0]
 ['Burnley' 'Tottenham' 7.0 ... 1.0 2.0 2.0]
 ['Sunderland' 'Newcastle' 8.0 ... 3.0 1.0 1.0]]


# After applying Label Encoding, the y-dependent variable changed:
#### **Home Win = 2**
#### **Away Win = 0**
#### **Draw = 1**


In [17]:
print(y_test)

[0 2 0 2 1 2 2 0 2 2 0 2 1 2 2 0 0 0 0 2 0 2 2 1 2 1 2 1 2 0 2 2 2 1 2 2 2
 2 0 0 1 0 0 0 1 1 0 2 0 0 1 2 2 2 1 2 1 1 1 1 1 2]


In [18]:
for i in range(len(X_test)):
  if y_test[i]==0:
    outcome=X_test[i][1]
  elif y_test[i]==1:
    outcome="Draw"
  elif y_test[i]==2:
    outcome=X_test[i][0]
  print(X_test[i][0]+ " vs "+X_test[i][1] +" = " + outcome)

Aston Villa vs Chelsea = Chelsea
Leeds vs Everton = Leeds
Newcastle vs Arsenal = Arsenal
Bournemouth vs Fulham = Bournemouth
Leeds vs Newcastle = Draw
Man United vs Sunderland = Man United
Crystal Palace vs Brentford = Crystal Palace
Burnley vs Newcastle = Newcastle
Arsenal vs Sunderland = Arsenal
Fulham vs Brentford = Fulham
Nott'm Forest vs Chelsea = Chelsea
Man City vs West Ham = Man City
West Ham vs Man City = Draw
Aston Villa vs West Ham = Aston Villa
Wolves vs Liverpool = Wolves
Brighton vs Arsenal = Arsenal
Wolves vs Man City = Man City
West Ham vs Aston Villa = Aston Villa
Wolves vs Brentford = Brentford
Aston Villa vs Man United = Aston Villa
Fulham vs Arsenal = Arsenal
Chelsea vs Wolves = Chelsea
Arsenal vs Brentford = Arsenal
Man City vs Brighton = Draw
Burnley vs Leeds = Burnley
Burnley vs Everton = Draw
Newcastle vs Man City = Newcastle
Crystal Palace vs Sunderland = Draw
Liverpool vs West Ham = Liverpool
Tottenham vs Newcastle = Newcastle
Nott'm Forest vs Leeds = Nott'm F

In [19]:
X_test_copy=X_test.copy()
y_test_copy=y_test.copy()

## **Applying Target Encoding**
*Applying target encoding at the first 2 columns (categories) that are "HomeTeam" and "AwayTeam"*

In [20]:
from sklearn.preprocessing import TargetEncoder
from sklearn.compose import ColumnTransformer
CT=ColumnTransformer(transformers=[('encoder',
                                   TargetEncoder(target_type='continuous'),
                                   categorical_features)],
                     remainder='passthrough')

X_train=np.array(CT.fit_transform(X_train,y_train))
X_test=np.array(CT.transform(X_test))

## **Random Forest Classification**
*This is the first model , which will make a prediction based on '100 estimators' and after that , an XGBoost model will be created and will compare the values*

In [21]:
from sklearn.ensemble import RandomForestClassifier
rf_classifier=RandomForestClassifier(n_estimators=150,criterion='gini')
rf_classifier.fit(X_train,y_train)

RandomForestClassifier(n_estimators=150)

In [25]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import GridSearchCV
parameters = {
    'n_estimators':[100, 150, 200, 250],
    'criterion': ['gini','entropy'],
    'max_features':['sqrt','log2'],
    'class_weight':['balanced','balanced_subsample']
}
tscv=TimeSeriesSplit()
grid_search=GridSearchCV(estimator=rf_classifier,param_grid=parameters,cv=tscv,verbose=2,scoring='accuracy',n_jobs=-1)
grid_search.fit(X_train,y_train)
print("Best Accuracy found: {:.2f} %".format(grid_search.best_score_*100))
print("Best parameters for RF Classifier: ",grid_search.best_params_)

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best Accuracy found: 46.34 %
Best parameters for RF Classifier:  {'class_weight': 'balanced', 'criterion': 'entropy', 'max_features': 'log2', 'n_estimators': 150}


In [26]:
rf_classifier=RandomForestClassifier(criterion='entropy',class_weight='balanced',max_features='log2',n_estimators=150)

In [28]:
rf_classifier.fit(X_train,y_train)

RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_features='log2', n_estimators=150)

In [29]:
y_pred=rf_classifier.predict_proba(X_test)

In [30]:
for i in range(len(X_test)):
  print(f'===== Predictions for {X_test_copy[i][0]} vs {X_test_copy[i][1]} =====')
  print(X_test_copy[i][0]," Probability to win: {:.2f} %".format(y_pred[i,2]*100))
  print(X_test_copy[i][1]," Probability to win: {:.2f} %".format(y_pred[i,0]*100))
  print("Draw: {:.2f}% ".format(y_pred[i,1]*100))
  if y_test[i]==0:
    outcome=X_test_copy[i][1]
  elif y_test[i]==1:
    outcome="Draw"
  elif y_test[i]==2:
    outcome=X_test_copy[i][0]
  print(f"Outcome: {outcome}")
  print("\n"*2)

===== Predictions for Aston Villa vs Chelsea =====
Aston Villa  Probability to win: 42.00 %
Chelsea  Probability to win: 23.33 %
Draw: 34.67% 
Outcome: Chelsea



===== Predictions for Leeds vs Everton =====
Leeds  Probability to win: 20.67 %
Everton  Probability to win: 6.67 %
Draw: 72.67% 
Outcome: Leeds



===== Predictions for Newcastle vs Arsenal =====
Newcastle  Probability to win: 39.33 %
Arsenal  Probability to win: 36.00 %
Draw: 24.67% 
Outcome: Arsenal



===== Predictions for Bournemouth vs Fulham =====
Bournemouth  Probability to win: 55.33 %
Fulham  Probability to win: 30.00 %
Draw: 14.67% 
Outcome: Bournemouth



===== Predictions for Leeds vs Newcastle =====
Leeds  Probability to win: 29.33 %
Newcastle  Probability to win: 21.33 %
Draw: 49.33% 
Outcome: Draw



===== Predictions for Man United vs Sunderland =====
Man United  Probability to win: 58.00 %
Sunderland  Probability to win: 32.67 %
Draw: 9.33% 
Outcome: Man United



===== Predictions for Crystal Palace vs Bren

## **XGBoost Classifier**

In [31]:
from xgboost import XGBClassifier
classifier=XGBClassifier()

In [34]:
xg_parameters={
    'n_estimators':[100,120,150],
    'max_depth':[3,4,5],
    'learning_rate':[0.01,0.05,0.1],
    'subsample':[0.8,1],
    'colsample_bytree':[0.8,1]
}
grid_search_xg=GridSearchCV(estimator=classifier,param_grid=xg_parameters,cv=tscv,n_jobs=-1,verbose=2,scoring='accuracy')
grid_search_xg.fit(X_train,y_train)
print("Best Accuracy found: {:.2f} %".format(grid_search_xg.best_score_*100))
print("Best parameters for Calibrated XGB Classifier: ",grid_search_xg.best_params_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Accuracy found: 46.34 %
Best parameters for Calibrated XGB Classifier:  {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 120, 'subsample': 1}


In [35]:
classifier=XGBClassifier(colsample_bytree=0.8,learning_rate=0.01,max_depth=3,n_estimators=120,subsample=1)

In [36]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_xgb=CalibratedClassifierCV(estimator=classifier,method='sigmoid',cv=5)
calibrated_xgb.fit(X_train,y_train)

CalibratedClassifierCV(cv=5,
                       estimator=XGBClassifier(base_score=None, booster=None,
                                               callbacks=None,
                                               colsample_bylevel=None,
                                               colsample_bynode=None,
                                               colsample_bytree=0.8,
                                               device=None,
                                               early_stopping_rounds=None,
                                               enable_categorical=False,
                                               eval_metric=None,
                                               feature_types=None,
                                               feature_weights=None, gamma=None,
                                               grow_policy=None,
                                               importance_type=None,
                                               interaction_constraints=None,
                                               learning_rate=0.01, max_bin=None,
                                               max_cat_threshold=None,
                                               max_cat_to_onehot=None,
                                               max_delta_step=None, max_depth=3,
                                               max_leaves=None,
                                               min_child_weight=None,
                                               missing=nan,
                                               monotone_constraints=None,
                                               multi_strategy=None,
                                               n_estimators=120, n_jobs=None,
                                               num_parallel_tree=None, ...))

In [37]:
y_pred_XG=calibrated_xgb.predict_proba(X_test)

In [38]:
for i in range(len(X_test)):
  print(f'===== Predictions for {X_test_copy[i][0]} vs {X_test_copy[i][1]} =====')
  print(X_test_copy[i][0]," Probability to win: {:.2f} %".format(y_pred_XG[i,2]*100))
  print(X_test_copy[i][1]," Probability to win: {:.2f} %".format(y_pred_XG[i,0]*100))
  print("Draw: {:.2f}% ".format(y_pred_XG[i,1]*100))
  if y_test[i]==0:
    outcome=X_test_copy[i][1]
  elif y_test[i]==1:
    outcome="Draw"
  elif y_test[i]==2:
    outcome=X_test_copy[i][0]
  print(f"Outcome: {outcome}")
  print("\n"*2)

===== Predictions for Aston Villa vs Chelsea =====
Aston Villa  Probability to win: 37.67 %
Chelsea  Probability to win: 28.74 %
Draw: 33.59% 
Outcome: Chelsea



===== Predictions for Leeds vs Everton =====
Leeds  Probability to win: 45.82 %
Everton  Probability to win: 27.03 %
Draw: 27.16% 
Outcome: Leeds



===== Predictions for Newcastle vs Arsenal =====
Newcastle  Probability to win: 40.99 %
Arsenal  Probability to win: 30.71 %
Draw: 28.30% 
Outcome: Arsenal



===== Predictions for Bournemouth vs Fulham =====
Bournemouth  Probability to win: 50.65 %
Fulham  Probability to win: 26.04 %
Draw: 23.31% 
Outcome: Bournemouth



===== Predictions for Leeds vs Newcastle =====
Leeds  Probability to win: 45.29 %
Newcastle  Probability to win: 27.53 %
Draw: 27.19% 
Outcome: Draw



===== Predictions for Man United vs Sunderland =====
Man United  Probability to win: 43.17 %
Sunderland  Probability to win: 29.20 %
Draw: 27.63% 
Outcome: Man United



===== Predictions for Crystal Palace vs Br

## **Making the Confusion Matrix and the Accuracy Score**

In [39]:
from sklearn.metrics import confusion_matrix, classification_report
print("Confusion Matrix of the RandomForest Classifier Model")
print(confusion_matrix(y_test,rf_classifier.predict(X_test)))
print(classification_report(y_test,rf_classifier.predict(X_test), zero_division=0))

print()
print("Confusion Matrix of the XGBoost Classifier Model")
print(confusion_matrix(y_test,calibrated_xgb.predict(X_test)))
print(classification_report(y_test,calibrated_xgb.predict(X_test), zero_division=0))


Confusion Matrix of the RandomForest Classifier Model
[[ 8  2  8]
 [ 6  3  7]
 [ 7  5 16]]
              precision    recall  f1-score   support

           0       0.38      0.44      0.41        18
           1       0.30      0.19      0.23        16
           2       0.52      0.57      0.54        28

    accuracy                           0.44        62
   macro avg       0.40      0.40      0.39        62
weighted avg       0.42      0.44      0.42        62


Confusion Matrix of the XGBoost Classifier Model
[[ 4  0 14]
 [ 5  0 11]
 [ 4  0 24]]
              precision    recall  f1-score   support

           0       0.31      0.22      0.26        18
           1       0.00      0.00      0.00        16
           2       0.49      0.86      0.62        28

    accuracy                           0.45        62
   macro avg       0.27      0.36      0.29        62
weighted avg       0.31      0.45      0.36        62



## **Applying k-Fold Cross Validation with StratifiedKFold**

In [40]:
from sklearn.model_selection import cross_val_score,StratifiedKFold
skf=StratifiedKFold(n_splits=5)
accuracy_rf=cross_val_score(estimator=rf_classifier,X=X_train,y=y_train,cv=skf)
print("Cross Validation Scores of RF Model: {}".format(accuracy_rf))
print("Mean Accuracy of Random Forest Classifier: {:.2f} %".format(accuracy_rf.mean()*100))
print("Standard Deviation of RF Classifier: {:.2f} %".format(accuracy_rf.std()*100))
print()
accuracy_XG=cross_val_score(estimator=calibrated_xgb,X=X_train,y=y_train,cv=skf)
print("Cross Validation Scores of XGBoost Model: {}".format(accuracy_XG))
print("Accuracy of XGBoost Classifier: {:.2f} %".format(accuracy_XG.mean()*100))
print("Standard Deviation of XGBoost Classifier: {:.2f} %".format(accuracy_XG.std()*100))

Cross Validation Scores of RF Model: [0.48       0.5        0.32653061 0.30612245 0.3877551 ]
Mean Accuracy of Random Forest Classifier: 40.01 %
Standard Deviation of RF Classifier: 7.84 %

Cross Validation Scores of XGBoost Model: [0.44       0.46       0.40816327 0.42857143 0.40816327]
Accuracy of XGBoost Classifier: 42.90 %
Standard Deviation of XGBoost Classifier: 1.98 %


## **Applying k-Fold Cross Validation with TimeSeriesSplit**

In [41]:
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
accuracy_rf=cross_val_score(estimator=rf_classifier,X=X_train,y=y_train,cv=tscv)
print("Cross Validation Scores of RF Model: {}".format(accuracy_rf))
print("Mean Accuracy of Random Forest Classifier: {:.2f} %".format(accuracy_rf.mean()*100))
print("Standard Deviation of RF Classifier: {:.2f} %".format(accuracy_rf.std()*100))
print()
accuracy_XG=cross_val_score(estimator=calibrated_xgb,X=X_train,y=y_train,cv=tscv)
print("Cross Validation Scores of XGBoost Model: {}".format(accuracy_XG))
print("Accuracy of XGBoost Classifier: {:.2f} %".format(accuracy_XG.mean()*100))
print("Standard Deviation of XGBoost Classifier: {:.2f} %".format(accuracy_XG.std()*100))

Cross Validation Scores of RF Model: [0.41463415 0.51219512 0.3902439  0.41463415 0.46341463]
Mean Accuracy of Random Forest Classifier: 43.90 %
Standard Deviation of RF Classifier: 4.36 %

Cross Validation Scores of XGBoost Model: [0.41463415 0.56097561 0.51219512 0.29268293 0.53658537]
Accuracy of XGBoost Classifier: 46.34 %
Standard Deviation of XGBoost Classifier: 9.88 %
